In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoModel, AutoTokenizer
from datasets import load_dataset
from collections import defaultdict
from tqdm import tqdm
import random
import os
import numpy as np
import json

In [ ]:

base_path = '/content/drive/My Drive/SNLI_Data'

if os.path.exists(base_path):
    print(f"Drive mounted. Found directory: {base_path}")
    print("Files:", os.listdir(base_path))


# Files
TRAIN_FILE = os.path.join(base_path, "snli_1.0_train.jsonl")
DEV_FILE = os.path.join(base_path, "snli_1.0_dev.jsonl")

MODEL_NAME = 'roberta-base'
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5
MAX_LEN = 70
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Running on device: {DEVICE}")

Drive mounted. Found directory: /content/drive/My Drive/SNLI_Data
Files: ['snli_1.0_dev.jsonl', 'snli_1.0_test.jsonl', 'snli_1.0_train.jsonl', 'roberta_cosine_epoch3.pth', 'roberta_cosine_epoch_1.pth', 'roberta_cosine_epoch_2.pth', 'roberta_cosine_epoch_3.pth']
Running on device: cuda


In [ ]:
class RobertaEmbeddingModel(nn.Module):
    def __init__(self, model_name='roberta-base'):
        super(RobertaEmbeddingModel, self).__init__()
        self.roberta = AutoModel.from_pretrained(model_name)

    def mean_pooling(self, model_output, attention_mask):

        token_embeddings = model_output.last_hidden_state
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1) # sum of embedding by multiplying embedding with mask and adding
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9) #counts how many real words + exception handling of 1 sentence

        return sum_embeddings / sum_mask

    def forward(self, input_ids, attention_mask):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        sentence_embeddings = self.mean_pooling(outputs, attention_mask)
        # normalize resulting vector
        return F.normalize(sentence_embeddings, p=2, dim=1)

In [ ]:
def CosineLoss(emb1, emb2, labels, margin=0.5):

    cosine_sim = F.cosine_similarity(emb1, emb2)


    pos_loss = 1 - cosine_sim

    # We want sim < margin, so loss = max(0, sim - margin)
    neg_loss = F.relu(cosine_sim - margin)


    pos_mask = (labels == 1).float() # boolean check used to keeps positive loss for labels =1 and neg loss for labels =-1
    neg_mask = (labels == -1).float()

    loss = (pos_mask * pos_loss) + (neg_mask * neg_loss)


    return loss.mean()

In [ ]:

def load_local_snli(file_path, limit=None):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")

    print(f"Reading {file_path}...")

    # read all lines
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()


    random.shuffle(lines)

    data = []
    for i, line in enumerate(lines):
        if limit and i >= limit: break

        row = json.loads(line)

        label = row.get('gold_label')

        if label not in ['entailment', 'contradiction']:
            continue

        binary_label = 1 if label == 'entailment' else -1

        data.append({
            'premise': row['sentence1'],
            'hypothesis': row['sentence2'],
            'label': binary_label
        })

    return data

class LocalSNLIPairDataset(Dataset):
    def __init__(self, raw_data, tokenizer, max_length):
        self.pairs = raw_data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        item = self.pairs[idx]
        text1 = item['premise']
        text2 = item['hypothesis']
        label = item['label']

        # Tokenize both sentences
        t1 = self.tokenizer(text1, padding='max_length', truncation=True, max_length=self.max_length, return_tensors='pt')
        t2 = self.tokenizer(text2, padding='max_length', truncation=True, max_length=self.max_length, return_tensors='pt')

        return {
            'input_ids1': t1['input_ids'].squeeze(0),
            'attention_mask1': t1['attention_mask'].squeeze(0),
            'input_ids2': t2['input_ids'].squeeze(0),
            'attention_mask2': t2['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.float)
        }

In [ ]:
def PredAndGetAccuracyAndVisualize(model, dataloader, device, tokenizer = None, threshold=0.5):
    model.eval()
    correct = 0
    total = 0


    with torch.no_grad():
        for batch in dataloader:
            ids1, mask1 = batch['input_ids1'].to(device), batch['attention_mask1'].to(device)
            ids2, mask2 = batch['input_ids2'].to(device), batch['attention_mask2'].to(device)
            labels = batch['label'].to(device)

            emb1, emb2 = model(ids1, mask1), model(ids2, mask2)
            sim = F.cosine_similarity(emb1, emb2)
            preds = torch.where(sim > threshold, 1, -1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            # visualization only for training
            if tokenizer:
                print("\n" + "=" * 115)
                print(f"{'PREMISE':<40} | {'HYPOTHESIS':<40} | {'TRUE':<10} | {'PRED':<10} | {'SCORE':<6}")
                print("=" * 115)

                decoded_s1 = tokenizer.batch_decode(ids1, skip_special_tokens=True)
                decoded_s2 = tokenizer.batch_decode(ids2, skip_special_tokens=True)

                for j in range(len(labels)):
                    lbl_str = "entailment" if labels[j] == 1 else "contradiction"
                    pred_str = "entailment" if preds[j] == 1 else "contradiction"
                    marker = "✅" if preds[j] == labels[j] else "Wrong!!!!!!!"

                    s1 = (decoded_s1[j][:37] + '..') if len(decoded_s1[j]) > 37 else decoded_s1[j]
                    s2 = (decoded_s2[j][:37] + '..') if len(decoded_s2[j]) > 37 else decoded_s2[j]

                    # if j < 2:
                    #     print(f"\n[DEBUG] Embedding 1 (Premise) [{emb1[j].shape}]:\n{emb1[j].cpu().numpy()}")
                    #     print(f"[DEBUG] Embedding 2 (Hypothesis) [{emb2[j].shape}]:\n{emb2[j].cpu().numpy()}")
                    #     print("-" * 50)

                    print(f"{s1:<40} | {s2:<40} | {lbl_str:<10} | {pred_str:<10} | {sim[j]:.4f} {marker}")


                print("=" * 115)

    return correct / total

In [ ]:
# Load Data
train_data = load_local_snli(TRAIN_FILE, limit=50000)
dev_data = load_local_snli(DEV_FILE, limit=1000)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_loader = DataLoader(LocalSNLIPairDataset(train_data, tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(LocalSNLIPairDataset(dev_data, tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)

model = RobertaEmbeddingModel(MODEL_NAME).to(DEVICE)

Reading /content/drive/My Drive/SNLI_Data/snli_1.0_train.jsonl...
Reading /content/drive/My Drive/SNLI_Data/snli_1.0_dev.jsonl...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Calculate how many layers to freeze
total_encoder_layers = len(model.roberta.encoder.layer)
freeze_ratio = 0.4
num_layers_to_freeze = int(total_encoder_layers * freeze_ratio)

print(f"Total Layers: {total_encoder_layers}")
print(f"Freezing Bottom {num_layers_to_freeze} Layers (Ratio: {freeze_ratio})")


# selecting modules to freeze
modules_to_freeze = [
    model.roberta.embeddings,
    *model.roberta.encoder.layer[:num_layers_to_freeze]
]

# freeze
for module in modules_to_freeze:
    for param in module.parameters():
        param.requires_grad = False

print("❄️ Freeze applied successfully.")

Total Layers: 12
Freezing Bottom 4 Layers (Ratio: 0.4)
❄️ Freeze applied successfully.


In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters:     {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

if total_params == trainable_params:
    print("STATUS: Full Fine-Tuning (All layers are being trained).")
else:
    print(f"STATUS: Frozen Layers Detected ({total_params - trainable_params:,} params frozen).")

Total Parameters:     124,645,632
Trainable Parameters: 57,293,568
STATUS: Frozen Layers Detected (67,352,064 params frozen).


In [ ]:
# """training"""


# # Margin=0.5 means: If pairs are dissimilar, push them until their cosine sim is below 0.5
# optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

# print("\nStarting Training...")
# for epoch in range(EPOCHS):
#     model.train()
#     loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

#     for batch in loop:
#         ids1, mask1 = batch['input_ids1'].to(DEVICE), batch['attention_mask1'].to(DEVICE)
#         ids2, mask2 = batch['input_ids2'].to(DEVICE), batch['attention_mask2'].to(DEVICE)
#         labels = batch['label'].to(DEVICE)

#         optimizer.zero_grad()
#         emb1, emb2 = model(ids1, mask1), model(ids2, mask2)

#         loss = CosineLoss(emb1, emb2, labels, margin=0.5)
#         loss.backward()
#         optimizer.step()

#         loop.set_postfix(loss=loss.item())

#     print("Validating...")
#     val_acc = PredAndGetAccuracyAndVisualize(model, val_loader, DEVICE, threshold=0.5)

#     print(f"Epoch {epoch+1} Results -> Accuracy: {val_acc*100:.2f}%")

#     torch.save(model.state_dict(), os.path.join(base_path, f"roberta_cosine_epoch_{epoch+1}.pth"))

# print("Training Complete.")


Starting Training...


Epoch 1/3: 100%|██████████| 1252/1252 [06:12<00:00,  3.36it/s, loss=0.216]


Validating...
Epoch 1 Results -> Accuracy: 69.43%


Epoch 2/3: 100%|██████████| 1252/1252 [06:09<00:00,  3.39it/s, loss=0.111]


Validating...
Epoch 2 Results -> Accuracy: 72.35%


Epoch 3/3: 100%|██████████| 1252/1252 [06:09<00:00,  3.39it/s, loss=0.0838]


Validating...
Epoch 3 Results -> Accuracy: 81.26%
Training Complete.


In [ ]:
# model = RobertaEmbeddingModel()
# model.to(DEVICE)

# print("Base RoBERTa model initialized.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Empty model created.


In [ ]:
save_path = "/content/drive/My Drive/SNLI_Data/roberta_cosine_epoch_3.pth"

# load our saved  state dictionary of the model
model.load_state_dict(torch.load(save_path, map_location=DEVICE))

# Set to Eval mode
model.eval()

print(f"Success! Weights loaded from {save_path}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Success! Weights loaded from /content/drive/My Drive/SNLI_Data/roberta_cosine_epoch_3.pth


In [ ]:
def display_test_results(model, tokenizer, file_path, device, num_samples=35):
    print(f"Loading {num_samples} samples from test file...")

    # load test data

    full_data = load_local_snli(file_path)


    samples = random.sample(full_data, num_samples)

    # tokenization and pass data to data loader
    test_dataset = LocalSNLIPairDataset(samples, tokenizer, MAX_LEN)
    test_loader = DataLoader(test_dataset, batch_size=num_samples, shuffle=False)


    acc = PredAndGetAccuracyAndVisualize(model, test_loader, device, tokenizer=tokenizer)

    print(f"\nAccuracy on these {num_samples} samples: {acc*100:.1f}%")

In [ ]:
testFile = os.path.join(base_path, "snli_1.0_test.jsonl")
display_test_results(model, tokenizer, testFile, DEVICE)


Loading 35 samples from test file...
Reading /content/drive/My Drive/SNLI_Data/snli_1.0_test.jsonl...

PREMISE                                  | HYPOTHESIS                               | TRUE       | PRED       | SCORE 
A person in a black and green outfit ..  | a colorfully dressed child rides a bi..  | entailment | entailment | 0.9820 ✅
The guitarist performs a rocking solo..  | the musician is performing               | entailment | entailment | 0.9379 ✅
An oriental woman cleaning wood outsi..  | A woman is cleaning wood.                | entailment | entailment | 0.9276 ✅
A man is looking at the camera posing..  | The man is looking at the camera.        | entailment | entailment | 0.9226 ✅
A girl bounces in a bounce house.        | A girl is bouncing around.               | entailment | entailment | 0.8847 ✅
A woman in a fur lined plaid purple j..  | A woman in a pink coat is watching ch..  | contradiction | entailment | 0.6902 Wrong!!!!!!!
A skateboarder is performing a grab t.

In [ ]:
# see some raw embeddings


text_A = "A soccer player runs across the field."
text_B = "A cat is meowing."

print(f"Sentence A: '{text_A}'")
print(f"Sentence B: '{text_B}'")
print("-" * 60)


inputs_A = tokenizer(text_A, return_tensors='pt').to(DEVICE)
inputs_B = tokenizer(text_B, return_tensors='pt').to(DEVICE)


model.eval()
with torch.no_grad():

    emb_A = model(inputs_A['input_ids'], inputs_A['attention_mask'])
    emb_B = model(inputs_B['input_ids'], inputs_B['attention_mask'])


print(f"\n Embedding 1 Shape: {emb_A.shape}]:")

print(emb_A[0][:20].cpu().numpy())

print(f"\n Embedding 2 Shape: {emb_B.shape}]:")
print(emb_B[0][:20].cpu().numpy())

print("-" * 60)

sim = torch.nn.functional.cosine_similarity(emb_A, emb_B)

print(f" Similarity Score: {sim.item():.4f}")

Sentence A: 'A soccer player runs across the field.'
Sentence B: 'A cat is meowing.'
------------------------------------------------------------

 Embedding 1 Shape: torch.Size([1, 768])]:
[ 0.02178921  0.00404971 -0.06745314  0.05633577 -0.0310501   0.01922823
 -0.03569144  0.06090726  0.03766518  0.02167317  0.01531646  0.04621787
 -0.01097013  0.02457219  0.02554262  0.00474382  0.05309941 -0.00586852
  0.05931685 -0.00987393]

 Embedding 2 Shape: torch.Size([1, 768])]:
[ 0.06134011  0.01472768  0.03613985  0.02900276  0.09203383  0.02602249
  0.00115934 -0.01096147 -0.00481859  0.02195304 -0.00091641 -0.09583077
  0.01579827 -0.00808971  0.03209802 -0.08519336 -0.07962971 -0.00616451
 -0.09933906 -0.04649574]
------------------------------------------------------------
 Similarity Score: -0.1107


In [ ]:
#runniing
# TEST_FILE = os.path.join(base_path, "snli_1.0_test.jsonl")

# display_test_results(model, tokenizer, TEST_FILE, num_samples=35)

Loading 35 random samples from test file...
----------------------------------------------------------------------------------------------------
PREMISE                                  | HYPOTHESIS                               | TRUE       | PRED       | SCORE 
----------------------------------------------------------------------------------------------------
A young girl is playing a musical ins..  | A boy plays air guitar while lip sync..  | contradiction | contradiction | 0.4972 ✅
A small dog runs to catch a ball.        | A little dog chases a ball.              | entailment | entailment | 0.9969 ✅
A person crossing a bridge with train..  | A person jumped off the side of the b..  | contradiction | entailment | 0.8470 Wrong
People shopping at an outside market     | People walk around an inside mall.       | contradiction | contradiction | 0.3493 ✅
Man sits at the doorway of barber sho..  | a man is seated                          | entailment | entailment | 0.6198 ✅
A basketbal

In [ ]:

# INFERENCE SECTION


def predict_similarity(text1, text2, model, tokenizer, device):
    model.eval()

    t1 = tokenizer(text1, padding='max_length', truncation=True, max_length=MAX_LEN, return_tensors='pt')
    t2 = tokenizer(text2, padding='max_length', truncation=True, max_length=MAX_LEN, return_tensors='pt')

    t1 = {k: v.to(device) for k, v in t1.items()}
    t2 = {k: v.to(device) for k, v in t2.items()}

    with torch.no_grad():
        emb1 = model(t1['input_ids'], t1['attention_mask'])
        emb2 = model(t2['input_ids'], t2['attention_mask'])

    cosine_sim = F.cosine_similarity(emb1, emb2).item()

    return cosine_sim

# Testing

print("\n--- Testing Model Performance ---")

# Case: Similar Sentences
s1 = "A soccer player is running across the field."
s2 = "A person is playing football outdoors."
sim = predict_similarity(s1, s2, model, tokenizer, DEVICE)

print(f"Sentence A: {s1}")
print(f"Sentence B: {s2}")
print(f"Cosine Sim: {sim:.4f} (High is good)")
print("-" * 30)

# Case: Different Sentences
s3 = "A man is sitting on a chair reading a book."
sim_diff = predict_similarity(s1, s3, model, tokenizer, DEVICE)

print(f"Sentence A: {s1}")
print(f"Sentence B: {s3}")
print(f"Cosine Sim: {sim_diff:.4f}")

if sim > sim_diff:
    print("\nSUCCESS: The model correctly identified the similar pair as closer!")
else:
    print("\nFAILURE: The model thinks the different pair is closer. Needs more training")


--- Testing Model Performance ---
Sentence A: A soccer player is running across the field.
Sentence B: A person is playing football outdoors.
Cosine Sim: 0.6407 (High is good)
------------------------------
Sentence A: A soccer player is running across the field.
Sentence B: A man is sitting on a chair reading a book.
Cosine Sim: 0.0688

SUCCESS: The model correctly identified the similar pair as closer!


In [ ]:
# # 5. Quick Inference Test
#     print("\n--- Inference Test ---")
#     model.eval()

#     test_sentences = ["The stock market crashed.", "Wall street is down.", "The sun is shining."]
#     inputs = tokenizer(test_sentences, return_tensors="pt", padding=True, truncation=True).to(device)

#     with torch.no_grad():
#         embeddings = model(inputs['input_ids'], inputs['attention_mask'])

#     # Compare 1st sentence with 2nd and 3rd
#     sim_1_2 = F.cosine_similarity(embeddings[0].unsqueeze(0), embeddings[1].unsqueeze(0))
#     sim_1_3 = F.cosine_similarity(embeddings[0].unsqueeze(0), embeddings[2].unsqueeze(0))

#     print(f"Sentence 1: {test_sentences[0]}")
#     print(f"Similarity with '{test_sentences[1]}': {sim_1_2.item():.4f}")
#     print(f"Similarity with '{test_sentences[2]}': {sim_1_3.item():.4f}")

In [ ]:
# def debug_model_internals(model, tokenizer, device):
#     print("\n🔍 --- DIAGNOSTIC REPORT ---")
#     model.eval()

#     # 1. Test Input Variety
#     s1 = "The dog runs."
#     s2 = "A spaceship flies."

#     print(f"\n1. checking Tokenizer for: '{s1}' vs '{s2}'")
#     t1 = tokenizer(s1, return_tensors='pt', padding=True).to(device)
#     t2 = tokenizer(s2, return_tensors='pt', padding=True).to(device)

#     print(f"   Input IDs 1: {t1['input_ids'][0].tolist()}")
#     print(f"   Input IDs 2: {t2['input_ids'][0].tolist()}")

#     if torch.equal(t1['input_ids'], t2['input_ids']):
#         print("   ❌ CRITICAL FAIL: Input IDs are identical! Tokenizer is broken.")
#         return
#     else:
#         print("   ✅ Input IDs are different.")

#     # 2. Check Embeddings
#     print("\n2. Checking Vector Outputs")
#     with torch.no_grad():
#         v1 = model(t1['input_ids'], t1['attention_mask'])
#         v2 = model(t2['input_ids'], t2['attention_mask'])

#     print(f"   Vector 1 (First 5 dims): {v1[0][:5].cpu().numpy()}")
#     print(f"   Vector 2 (First 5 dims): {v2[0][:5].cpu().numpy()}")

#     # Check if vectors are identical
#     if torch.allclose(v1, v2, atol=1e-4):
#         print("   ❌ CRITICAL FAIL: Model outputs identical vectors for different text.")
#         print("      CAUSE: Model Collapse. The weights have degraded.")
#     else:
#         print("   ✅ Vectors are different.")

#     # 3. Check Similarity
#     sim = F.cosine_similarity(v1, v2).item()
#     print(f" \n3.  Cosine Similarity: {sim:.5f}")
#     if sim > 0.99:
#         print("   ⚠️ WARNING: Similarity is effectively 1.0. The model has collapsed.")

# # Run the debug
# debug_model_internals(model, tokenizer, DEVICE)

# # 4. Check Dataset Labels (To ensure we aren't training only on Positives)
# print("\n4. Checking Data Distribution")
# try:
#     pos_count = sum(1 for x in train_data if x['label'] == 1)
#     neg_count = sum(1 for x in train_data if x['label'] == -1)
#     print(f"   Training Data: {len(train_data)} total rows")
#     print(f"   Positives (1): {pos_count}")
#     print(f"   Negatives (-1): {neg_count}")

#     if neg_count == 0:
#         print("   ❌ CRITICAL FAIL: You have ZERO negative samples. The model learned to always predict 'Similar'.")
# except NameError:
#     print("   (Skipping data check, 'train_data' variable not found in memory)")


🔍 --- DIAGNOSTIC REPORT ---

1. checking Tokenizer for: 'The dog runs.' vs 'A spaceship flies.'
   Input IDs 1: [0, 133, 2335, 1237, 4, 2]
   Input IDs 2: [0, 250, 39881, 16016, 4, 2]
   ✅ Input IDs are different.

2. Checking Vector Outputs
   Vector 1 (First 5 dims): [ 0.00577697 -0.01187757 -0.04833732  0.03920275 -0.01578885]
   Vector 2 (First 5 dims): [ 0.01326255 -0.01037327 -0.00655585 -0.03196639  0.0004344 ]
   ✅ Vectors are different.
 
3.  Cosine Similarity: -0.03733

4. Checking Data Distribution
   Training Data: 20032 total rows
   Positives (1): 10020
   Negatives (-1): 10012
